# 07 - Informes por proyecto

Este notebook genera un informe Markdown y HTML dentro de cada carpeta `analysis_outputs`.

El informe no es la memoria final, pero sirve como borrador y como material de apoyo.

In [9]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LEVEL_ORDER = {"A1": 1, "A2": 2, "B1": 3, "B2": 4, "C1": 5, "C2": 6}
LEVEL_ORDER_INV = {v: k for k, v in LEVEL_ORDER.items()}
RADON_GRADE_ORDER = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6}
RADON_GRADE_ORDER_INV = {v: k for k, v in RADON_GRADE_ORDER.items()}

def clean_file_name(path):
    """Devuelve un nombre de fichero comparable entre herramientas."""
    return Path(str(path)).name

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def detect_json_type(path):
    """Intenta detectar si un JSON parece de Radon o de PyCEFR."""
    try:
        data = load_json(path)
    except Exception:
        return "unknown"
    # Buscar un registro de ejemplo dentro del árbol project/file/[records]
    for project, files in data.items():
        if not isinstance(files, dict):
            continue
        for file_path, records in files.items():
            if isinstance(records, list) and records:
                rec = records[0]
                if isinstance(rec, dict):
                    if {"Class", "Start Line", "End Line", "Level"}.issubset(set(rec.keys())):
                        return "pycefr"
                    if {"type", "rank", "complexity", "lineno", "endline"}.issubset(set(rec.keys())):
                        return "radon"
    return "unknown"

In [10]:
def read_json_safe(path):
    if not path.exists():
        return {}

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def df_head_md(path, n=10):
    if not path.exists():
        return "No disponible."

    df = pd.read_csv(path)

    if df.empty:
        return "Tabla vacía."

    return df.head(n).to_markdown(index=False)


def make_report(case_dir):
    out = case_dir / "analysis_outputs"

    summary = read_json_safe(out / "summary.json")

    interesting_path = out / "interesting_cases.csv"
    py_path = out / "pycefr_constructs.csv"
    ra_path = out / "radon_functions.csv"

    py = pd.read_csv(py_path) if py_path.exists() else pd.DataFrame()
    ra = pd.read_csv(ra_path) if ra_path.exists() else pd.DataFrame()
    it = pd.read_csv(interesting_path) if interesting_path.exists() else pd.DataFrame()

    if not it.empty and "case_type" in it.columns:
        interesting_filtered = it[it["case_type"] != "Sin patrón destacado"]
    else:
        interesting_filtered = pd.DataFrame()

    md = []

    md.append(f"# Informe del caso: {case_dir.name}")
    md.append("## Resumen general")

    md.append(f"- Constructos PyCEFR detectados: {summary.get('n_pycefr_constructs', 0)}")
    md.append(f"- Clases PyCEFR distintas: {summary.get('n_pycefr_classes', 0)}")
    md.append(f"- Nivel máximo PyCEFR: {summary.get('max_pycefr_level')}")
    md.append(f"- Funciones Radon detectadas: {summary.get('n_radon_functions', 0)}")
    md.append(f"- Complejidad máxima Radon: {summary.get('max_radon_complexity')}")
    md.append(f"- Calificación máxima de Radon: {summary.get('max_radon_grade')}")
    md.append(f"- Casos interesantes detectados: {summary.get('n_interesting_cases', 0)}")

    md.append("## Interpretación inicial")

    if summary.get("n_interesting_cases", 0):
        md.append(
            "Este caso contiene funciones donde las herramientas muestran patrones de "
            "coincidencia o discrepancia. Estos ejemplos son candidatos para comentarse "
            "en la memoria."
        )
    else:
        md.append(
            "En este caso no se han detectado grandes discrepancias automáticas con las "
            "reglas actuales. Aun así, puede ser útil revisar las funciones más complejas "
            "o los constructos de nivel más alto."
        )

    md.append("## Constructos PyCEFR más frecuentes")

    if not py.empty and "class" in py.columns:
        top_classes = (
            py["class"]
            .value_counts()
            .head(10)
            .reset_index()
        )

        top_classes.columns = ["class", "count"]
        md.append(top_classes.to_markdown(index=False))
    else:
        md.append("No hay datos de PyCEFR.")

    md.append("## Funciones más complejas según Radon")

    if not ra.empty and "complexity" in ra.columns:
        cols = [
            c for c in ["file_name", "file_clean", "name", "complexity", "radon_grade", "lineno", "endline"]
            if c in ra.columns
        ]

        top_radon = (
            ra
            .sort_values("complexity", ascending=False)
            .head(10)
        )

        md.append(top_radon[cols].to_markdown(index=False))
    else:
        md.append("No hay datos de Radon.")

    md.append("## Casos interesantes")

    if not interesting_filtered.empty:
        cols = [
            c for c in [
                "file_name",
                "file_clean",
                "function",
                "name",
                "complexity",
                "radon_grade",
                "max_level",
                "n_pycefr_constructs",
                "case_type",
            ]
            if c in interesting_filtered.columns
        ]

        md.append(interesting_filtered[cols].head(15).to_markdown(index=False))
    else:
        md.append("No se han detectado casos destacados con las reglas actuales.")

    md.append("## Figuras generadas")

    figures = sorted(out.glob("*.png"))

    if figures:
        for fig in figures:
            md.append(f"![{fig.stem}]({fig.name})")
    else:
        md.append("No hay figuras generadas para este caso.")

    md.append("## Nota para revisión manual")
    md.append(
        "Este informe se genera automáticamente. Las conclusiones finales deben revisarse "
        "manualmente para comprobar que los ejemplos seleccionados son representativos y "
        "adecuados para la memoria."
    )

    report_text = "\n\n".join(md)

    (out / "report.md").write_text(report_text, encoding="utf-8")

    try:
        import markdown
        html_body = markdown.markdown(report_text, extensions=["tables"])
    except Exception:
        html_body = "<pre>" + report_text.replace("&", "&amp;").replace("<", "&lt;") + "</pre>"

    html = f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>Informe {case_dir.name}</title>
<style>
body {{ font-family: Arial, sans-serif; max-width: 1000px; margin: 40px auto; line-height: 1.5; }}
table {{ border-collapse: collapse; width: 100%; margin: 1em 0; }}
th, td {{ border: 1px solid #ddd; padding: 6px; }}
th {{ background: #f2f2f2; }}
img {{ max-width: 900px; display: block; margin: 20px 0; }}
</style>
</head>
<body>{html_body}</body>
</html>"""

    (out / "report.html").write_text(html, encoding="utf-8")

    return out / "report.md", out / "report.html"

In [13]:

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_case_folders():
    """
    Devuelve las carpetas de casos dentro de data/processed.
    Cada subcarpeta se considera un caso/proyecto.
    """
    if not PROCESSED_DIR.exists():
        print("No existe:", PROCESSED_DIR)
        return []

    return sorted([p for p in PROCESSED_DIR.iterdir() if p.is_dir()])

In [14]:
generated = []

for case_dir in find_case_folders():
    out = case_dir / "analysis_outputs"

    if out.exists():
        generated.append(make_report(case_dir))

for md, html in generated:
    print("Generado:", md)
    print("Generado:", html)

Generado: /home/juan/Documents/Analisis-CC-PyCEFR/data/processed/python-beginner-programming-exercises/analysis_outputs/report.md
Generado: /home/juan/Documents/Analisis-CC-PyCEFR/data/processed/python-beginner-programming-exercises/analysis_outputs/report.html


## Comentario para la memoria

Los informes por proyecto permiten revisar rápidamente los resultados de cada repositorio. En la memoria se usarán como apoyo para seleccionar ejemplos y extraer conclusiones, no como sustituto del análisis crítico.